In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats
import numpy as np

In [ ]:
game_data = pd.read_csv("data/processed/steam_games.csv")
game_data.columns
game_data.head()
print(list(game_data.columns))

In [ ]:
# the most successful games by overall number of players
# it looks like players data is from the moment we queried the API, at least when it comes to players
successful_games = game_data.nlargest(10, 'players')
successful_games
# Print just the name and players columns
print(successful_games[['name', 'players']])

In [ ]:
import matplotlib.pyplot as plt

# Create the bar chart
successful_games.plot.bar(x='name', y='players', rot=45, legend=False)

# Rotate by 45 degrees and align them properly under the bars
plt.xticks(rotation=45, ha='right')

# Add titles and labels
plt.title('Most Successful Games by Player Count')
plt.xlabel('Game Name')
plt.ylabel('Number of Players')

# Adjust layout so labels don't get cut off at the bottom
plt.tight_layout()

# Display the chart
plt.show()

In [ ]:
# Most successful games by reviews
successful_games_by_reviews = game_data.nlargest(10, ['reviews.score', 'reviews.total'])
successful_games_by_reviews
# Print just the name and players columns
print(successful_games_by_reviews[['name', 'reviews.score', 'reviews.total']])

In [ ]:
# Create the bar chart
successful_games_by_reviews.plot.bar(x='name', y='reviews.total', rot=45, legend=False)

# Rotate by 45 degrees and align them properly under the bars
plt.xticks(rotation=45, ha='right')

# Add titles and labels
plt.title('Most Successful Games by Review Count')
plt.xlabel('Game Name')
plt.ylabel('Number of Reviews')

# Adjust layout so labels don't get cut off at the bottom
plt.tight_layout()

# Display the chart
plt.show()

In [ ]:
# the most successful games by overall number of players
# it looks like all the data is from the last 30 days, at least when it comes to players
successful_games = game_data.nlargest(10, 'peakalltime')
successful_games
# Print just the name and players columns
print(successful_games[['name', 'peakalltime']])

In [ ]:
# Main columns indicating player engagement
engagement_cols = [
    'avg30d',
    'peak30d',
    'players24h',
    'hoursplayed30d',
    'players',
    'peakalltime'
]

# Basic summary
game_data[engagement_cols].describe().T

In [ ]:
zero_avg = game_data[game_data['avg30d'] == 0]

zero_avg[['name', 'avg30d', 'players24h', 'peak30d']].head(30)

# Compare player metrics for zero-average games
zero_avg[
    ['avg30d', 'players24h', 'peak30d', 'players', 'peakalltime']
].describe()

In [ ]:
# Distribution of 30-day average player counts
plt.figure(figsize=(10, 6))

sns.histplot(
    data=game_data,
    x='avg30d',
    bins=100
)

plt.title('Distribution of 30-Day Average Player Count')
plt.xlabel('30-Day Average Players')
plt.ylabel('Number of Games')

plt.tight_layout()
plt.show()

# Above chart on a log scale because it's really bad
game_data['log_avg30d'] = np.log10(game_data['avg30d'] + 1)

plt.figure(figsize=(10, 6))

sns.histplot(
    data=game_data,
    x='log_avg30d',
    bins=50
)

plt.title('Distribution of 30-Day Average Players (Log Scale)')
plt.xlabel('log10(30-Day Average Players + 1)')
plt.ylabel('Number of Games')

plt.tight_layout()
plt.show()

In [ ]:
# How many games have at least one current player?
print("avg30d > 0:", (game_data['avg30d'] > 0).sum())

# How many have at least one player historically?
print("peakalltime > 0:", (game_data['peakalltime'] > 0).sum())

# How many have historical players but currently have zero?
historically_active_currently_zero = (
    (game_data['peakalltime'] > 0) &
    (game_data['avg30d'] == 0)
)

print(
    "Historical players but avg30d = 0:",
    historically_active_currently_zero.sum()
)

In [ ]:
# Create meaningful player-count categories
def categorize_players(x):
    if x == 0:
        return '0'
    elif x <= 10:
        return '1–10'
    elif x <= 100:
        return '11–100'
    elif x <= 1000:
        return '101–1,000'
    elif x <= 10000:
        return '1,001–10,000'
    elif x <= 100000:
        return '10,001–100,000'
    else:
        return '100,000+'

game_data['player_category'] = (
    game_data['avg30d']
    .apply(categorize_players)
)

category_order = [
    '0',
    '1–10',
    '11–100',
    '101–1,000',
    '1,001–10,000',
    '10,001–100,000',
    '100,000+'
]

category_counts = (
    game_data['player_category']
    .value_counts()
    .reindex(category_order)
)

print(category_counts)

category_percentages = (
    category_counts / category_counts.sum() * 100
)

print(
    category_percentages.round(1).to_string()
)

plt.figure(figsize=(11, 6))

sns.barplot(
    x=category_counts.index,
    y=category_counts.values,
    color='steelblue'
)

plt.title('Steam Games by 30-Day Average Player Count')
plt.xlabel('30-Day Average Players')
plt.ylabel('Number of Games')

plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Remove games where avg30d is missing
analysis_data = game_data.dropna(
    subset=['avg30d']
).copy()

# Calculate the 90th percentile
success_threshold = analysis_data['avg30d'].quantile(0.90)

print(
    f"90th percentile of average 30-day players: "
    f"{success_threshold:,.0f}"
)

# Define highly successful games
analysis_data['highly_successful'] = (
    analysis_data['avg30d'] >= success_threshold
)

print("\nNumber of highly successful games:")
print(
    analysis_data['highly_successful'].sum()
)

print("\nPercentage highly successful:")
print(
    analysis_data['highly_successful'].mean() * 100
)

In [ ]:
# Lets see what the 'most successful' games are based on player
successful_games = (
    analysis_data[
        analysis_data['highly_successful']
    ]
    .sort_values('avg30d', ascending=False)
)

print(
    successful_games[
        ['name', 'avg30d', 'peak30d', 'peakalltime']
    ].head(30).to_string(index=False)
)

print(
    "Minimum avg30d among highly successful games:",
    successful_games['avg30d'].min()
)

print(
    "Maximum avg30d among highly successful games:",
    successful_games['avg30d'].max()
)

In [ ]:
# Checking what data has made it over here
candidate_variables = [
    'price',
    'positivepercent',
    'metacritic',
    'recommendations',
    'reviews.total',
    'reviews.score',
    'release_age',
    'isfree',
    'dlccount',
    'windows',
    'mac',
    'linux'
]

available_variables = [
    col for col in candidate_variables
    if col in analysis_data.columns
]

print(available_variables)

In [ ]:
# Lets see what distinguishes them and indicates people will come back
comparison_variables = [
    'metacritic',
    'recommendations',
    'reviews.total',
    'reviews.score',
    'isfree',
    'dlccount'
]

comparison = (
    analysis_data
    .groupby('highly_successful')[comparison_variables]
    .median()
    .T
)

comparison.columns = [
    'Other Games',
    'Highly Successful'
]

comparison['Difference'] = (
    comparison['Highly Successful']
    - comparison['Other Games']
)

comparison


In [ ]:
# The following boxplots are for displaying differences between large player
# counts and others in varying different things.
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=analysis_data,
    x='highly_successful',
    y='metacritic'
)

plt.xticks(
    [0, 1],
    ['Other Games', 'Highly Successful']
)

plt.title('Metacritic Scores: Successful vs. Other Games')
plt.xlabel('')
plt.ylabel('Metacritic Score')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=analysis_data,
    x='highly_successful',
    y='reviews.score'
)

plt.xticks(
    [0, 1],
    ['Other Games', 'Highly Successful']
)

plt.title('Steam Review Scores: Successful vs. Other Games')
plt.xlabel('')
plt.ylabel('Review Score')

plt.tight_layout()
plt.show()

In [ ]:
analysis_data['log_recommendations'] = np.log10(
    analysis_data['recommendations'].clip(lower=0) + 1
)

plt.figure(figsize=(8, 6))

sns.boxplot(
    data=analysis_data,
    x='highly_successful',
    y='log_recommendations'
)

plt.xticks(
    [0, 1],
    ['Other Games', 'Highly Successful']
)

plt.title('Recommendations: Successful vs. Other Games')
plt.xlabel('')
plt.ylabel('log10(Recommendations + 1)')

plt.tight_layout()
plt.show()

In [ ]:
analysis_data['log_reviews'] = np.log10(
    analysis_data['reviews.total'].clip(lower=0) + 1
)

plt.figure(figsize=(8, 6))

sns.boxplot(
    data=analysis_data,
    x='highly_successful',
    y='log_reviews'
)

plt.xticks(
    [0, 1],
    ['Other Games', 'Highly Successful']
)

plt.title('Review Count: Successful vs. Other Games')
plt.xlabel('')
plt.ylabel('log10(Total Reviews + 1)')

plt.tight_layout()
plt.show()

In [ ]:
free_comparison = (
    analysis_data
    .groupby('highly_successful')['isfree']
    .mean()
    * 100
)

free_comparison.index = [
    'Other Games',
    'Highly Successful'
]

print(
    free_comparison.round(1)
)

plt.figure(figsize=(7, 5))

sns.barplot(
    x=free_comparison.index,
    y=free_comparison.values,
    color='steelblue'
)

plt.title('Percentage of Games That Are Free-to-Play')
plt.xlabel('')
plt.ylabel('Percentage of Games')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=analysis_data,
    x='highly_successful',
    y='dlccount'
)

plt.xticks(
    [0, 1],
    ['Other Games', 'Highly Successful']
)

plt.title('DLC Count: Successful vs. Other Games')
plt.xlabel('')
plt.ylabel('Number of DLC Items')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

# Mann-Whitney U Test to see if there's a correlation
test_variables = [
    'metacritic',
    'recommendations',
    'reviews.total',
    'reviews.score',
    'dlccount'
]

test_results = []

for variable in test_variables:

    successful = analysis_data.loc[
        analysis_data['highly_successful'],
        variable
    ].dropna()

    other = analysis_data.loc[
        ~analysis_data['highly_successful'],
        variable
    ].dropna()

    statistic, p_value = mannwhitneyu(
        successful,
        other,
        alternative='two-sided'
    )

    test_results.append({
        'Variable': variable,
        'Successful Median': successful.median(),
        'Other Median': other.median(),
        'p-value': p_value
    })

statistical_comparisons = pd.DataFrame(
    test_results
)

statistical_comparisons

In [ ]:
# Correlation matrix to see what variables are associated with player engagement
correlation_variables = [
    'log_avg30d',
    'metacritic',
    'recommendations',
    'reviews.total',
    'reviews.score',
    'isfree',
    'dlccount'
]

correlation_variables = [
    col for col in correlation_variables
    if col in analysis_data.columns
]

correlation_data = analysis_data[
    correlation_variables
].copy()

correlation_matrix = correlation_data.corr()

correlation_matrix

In [ ]:
# Most important to look at is the log_avg30d column
plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1
)

plt.title(
    'Correlation Between Game Characteristics and Player Engagement'
)

plt.tight_layout()
plt.show()

In [ ]:
# The most compelling of the comparisons
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=analysis_data,
    x='log_recommendations',
    y='log_avg30d',
    alpha=0.25,
    s=25
)

plt.title(
    'Recommendations vs. Player Engagement'
)

plt.xlabel('log10(Recommendations + 1)')
plt.ylabel('log10(30-Day Average Players + 1)')

plt.tight_layout()
plt.show()

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# MODEL A - Characteristics associated with engagement
# Define the data
model_a_variables = [
    'metacritic',
    'reviews.score',
    'isfree',
    'dlccount'
]

model_a_variables = [
    col for col in model_a_variables
    if col in analysis_data.columns
]

model_a_data = analysis_data[
    ['log_avg30d'] + model_a_variables
].copy()

model_a_data = model_a_data.dropna(
    subset=['log_avg30d']
)

X = model_a_data[model_a_variables]
y = model_a_data['log_avg30d']

# Split the data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Build the model
model_a = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    ),
    (
        'scaler',
        StandardScaler()
    ),
    (
        'regression',
        LinearRegression()
    )
])

model_a.fit(
    X_train,
    y_train
)

In [ ]:
# Model A evaluation
y_pred_a = model_a.predict(X_test)

r2_a = r2_score(
    y_test,
    y_pred_a
)

rmse_a = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_a
    )
)

print("Model A")
print("R²:", round(r2_a, 3))
print("RMSE:", round(rmse_a, 3))

In [ ]:
# Examining the coefficients
model_a_regression = (
    model_a.named_steps['regression']
)

model_a_coefficients = pd.DataFrame({
    'Variable': model_a_variables,
    'Coefficient': model_a_regression.coef_
})

model_a_coefficients['Absolute Effect'] = (
    model_a_coefficients['Coefficient'].abs()
)

model_a_coefficients = (
    model_a_coefficients
    .sort_values(
        'Absolute Effect',
        ascending=False
    )
)

model_a_coefficients

In [ ]:
# Visualizing the above
plt.figure(figsize=(9, 6))

sns.barplot(
    data=model_a_coefficients,
    x='Coefficient',
    y='Variable',
    color='steelblue'
)

plt.axvline(
    0,
    color='black',
    linewidth=0.8
)

plt.title(
    'Characteristics Associated with Steam Player Engagement'
)

plt.xlabel('Standardized regression coefficient')
plt.ylabel('')

plt.tight_layout()
plt.show()

# The graph below shows that games with higher review scores tend to have
# higher player engagement after accounting for other characteristics

In [ ]:
# MODEL B - Popularity-related variables
model_b_variables = [
    'metacritic',
    'reviews.score',
    'isfree',
    'dlccount',
    'reviews.total',
    'recommendations'
]

model_b_variables = [
    col for col in model_b_variables
    if col in analysis_data.columns
]

model_b_data = analysis_data[
    ['log_avg30d'] + model_b_variables
].copy()

model_b_data = model_b_data.dropna(
    subset=['log_avg30d']
)

# Log-transform the highly skewed variables
model_b_data['log_reviews'] = np.log10(
    model_b_data['reviews.total'].clip(lower=0) + 1
)

model_b_data['log_recommendations'] = np.log10(
    model_b_data['recommendations'].clip(lower=0) + 1
)

model_b_variables = [
    'metacritic',
    'reviews.score',
    'isfree',
    'dlccount',
    'log_reviews',
    'log_recommendations'
]

X = model_b_data[model_b_variables]
y = model_b_data['log_avg30d']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model_b = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    ),
    (
        'scaler',
        StandardScaler()
    ),
    (
        'regression',
        LinearRegression()
    )
])

model_b.fit(
    X_train,
    y_train
)

y_pred_b = model_b.predict(X_test)

r2_b = r2_score(
    y_test,
    y_pred_b
)

rmse_b = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_b
    )
)

print("Model B")
print("R²:", round(r2_b, 3))
print("RMSE:", round(rmse_b, 3))

model_b_regression = (
    model_b.named_steps['regression']
)

model_b_coefficients = pd.DataFrame({
    'Variable': model_b_variables,
    'Coefficient': model_b_regression.coef_
})

model_b_coefficients['Absolute Effect'] = (
    model_b_coefficients['Coefficient'].abs()
)

model_b_coefficients = (
    model_b_coefficients
    .sort_values(
        'Absolute Effect',
        ascending=False
    )
)

display(model_b_coefficients)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=model_b_coefficients,
    x='Coefficient',
    y='Variable',
    color='darkorange'
)

plt.axvline(
    0,
    color='black',
    linewidth=0.8
)

plt.title(
    'Regression Coefficients for Steam Player Engagement'
)

plt.xlabel('Standardized regression coefficient')
plt.ylabel('')

plt.tight_layout()
plt.show()


In [ ]:
# Compare the models
model_comparison = pd.DataFrame({
    'Model': [
        'Model A: Game characteristics',
        'Model B: + Reviews & Recommendations'
    ],
    'R²': [
        r2_a,
        r2_b
    ],
    'RMSE': [
        rmse_a,
        rmse_b
    ]
})

model_comparison

In [ ]:
# Actual vs Predicted Player Engagement
plt.figure(figsize=(8, 8))

sns.scatterplot(
    x=y_test,
    y=y_pred_b,
    alpha=0.3
)

# Perfect prediction line
min_value = min(
    y_test.min(),
    y_pred_b.min()
)

max_value = max(
    y_test.max(),
    y_pred_b.max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle='--',
    color='red'
)

plt.title(
    'Actual vs. Predicted Player Engagement'
)

plt.xlabel(
    'Actual log10(30-Day Average Players + 1)'
)

plt.ylabel(
    'Predicted log10(30-Day Average Players + 1)'
)

plt.tight_layout()
plt.show()

# Main takeaway 1: Metacritic, review score, free-to-play status, and DLC count
# explain about 16% of the variation in player engagement.
# Main takeaway 2: Reviews and recommendations are much more strongly associated
# with continued engagement, with review count being the strongest predictor in model B
# Overall: Games with larger review and recommendation footprints are strongly
# associated with higher current player engagement.

In [ ]:
# Check correlations among Model B predictors

model_b_correlation = model_b_data[
    model_b_variables
].corr()

plt.figure(figsize=(9, 7))

sns.heatmap(
    model_b_correlation,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1
)

plt.title('Correlation Among Regression Predictors')

plt.tight_layout()
plt.show()